# 05 — Entraînement MLlib & MLflow 

**NYC Taxi Demand Prediction · Microsoft Fabric · Spark MLlib**

**Entrée** : `gold.taxi_demand_features` (Delta) · **Sorties** : métriques MLflow, modèle Spark dans OneLake, prévisions de test optionnelles.

**Modèles comparés** : baseline zone × jour × heure, Random Forest MLlib, GBT MLlib (ce dernier n'est **pas** XGBoost).

**Principe** : sélectionner uniquement sur **validation**, puis tester une seule fois sur novembre–décembre. Aucune conversion Pandas pour l'entraînement.

> Cette version teste l'écriture d'artefacts MLflow dans une **nouvelle expérience** avant l'entraînement, puis permet de continuer même si seul l'upload MLflow échoue. Ne pas confondre upload d'artefacts et MLflow tracking. Des prédictions historiques fondées sur la météo historique ne prouvent pas encore la performance d'une prévision météo opérationnelle. Attention aussi aux heures locales naïves/DST et aux éventuels biais du nettoyage Silver.

## 1. Configuration — attacher le Lakehouse par défaut

In [ ]:
import json
import math
import tempfile
import os
from pathlib import Path
import mlflow
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import RandomForestRegressor, GBTRegressor
from pyspark.ml.evaluation import RegressionEvaluator

GOLD_TABLE = "gold.taxi_demand_features"
# On crée une nouvelle expérience pour contourner un éventuel problème propre à l'ancienne.
# Si nécessaire, remplacer par "nyc_taxi_mllib_permission_diagnostic" (test réussi précédemment).
EXPERIMENT_NAME = "nyc_taxi_demand_mllib_v2"
FEATURES = [
    "taxi_zone_id", "month", "day_of_month", "day_of_week", "hour_of_day", "is_weekend",
    "hour_sin", "hour_cos", "dow_sin", "dow_cos", "month_sin", "month_cos",
    "weather_temperature_2m", "weather_relative_humidity_2m",
    "weather_precipitation", "weather_rain", "weather_snowfall",
    "weather_wind_speed_10m", "weather_weather_code"
]
TARGET = "pickup_count"
ZONES = {230, 161, 237, 186, 87}
TRAIN_END = "2025-09-01"
VALIDATION_END = "2025-11-01"
SEED = 42
MODEL_PATH = "Files/models/nyc_taxi_mllib_best_v2"
WRITE_TEST_PREDICTIONS = True
# Ne pas activer d'autologging de modèles : l'upload de Spark models est distinct de celui des petits fichiers.
mlflow.autolog(disable=True)
print("Spark:", spark.version, "MLflow:", mlflow.__version__)
print("Tracking URI Fabric:", mlflow.get_tracking_uri())

## 2. Tester MLflow — paramètres, métriques **et artefact**

In [ ]:
experiment = mlflow.set_experiment(EXPERIMENT_NAME)
ARTIFACT_UPLOAD_OK = False
try:
    with tempfile.TemporaryDirectory() as tmp:
        file_path = Path(tmp) / "mlflow_probe.txt"
        file_path.write_text("Microsoft Fabric artifact upload test", encoding="utf-8")
        with mlflow.start_run(run_name="00_artifact_smoke_test") as run:
            mlflow.log_param("framework", "Spark MLlib")
            mlflow.log_metric("probe_value", 1.0)
            mlflow.log_artifact(str(file_path), artifact_path="diagnostic")
        ARTIFACT_UPLOAD_OK = True
        print("OK : tracking + upload d'artefact fonctionnels; run:", run.info.run_id)
except Exception as exc:
    print("AVERTISSEMENT : artefact MLflow inaccessible, type:", type(exc).__name__)
    print(str(exc)[:600])
    print("On peut poursuivre l'entraînement et sauvegarder le modèle dans OneLake.")

print("Experience ID:", experiment.experiment_id)
print("ARTIFACT_UPLOAD_OK:", ARTIFACT_UPLOAD_OK)

## 3. Charger et contrôler la table Gold

L'ordre temporel, les zones, les valeurs manquantes et l'unicité `(taxi_zone_id, hour_local)` sont vérifiés **avant** l'entraînement.

In [ ]:
df_gold = spark.table(GOLD_TABLE)
required = set(FEATURES + [TARGET, "hour_local"])
missing_cols = sorted(required - set(df_gold.columns))
assert not missing_cols, f"Colonnes Gold absentes : {missing_cols}"
assert df_gold.count() > 0, "Gold est vide"
assert df_gold.select("taxi_zone_id", "hour_local").distinct().count() == df_gold.count(), "Doublons zone-heure"
actual_zones = {int(r[0]) for r in df_gold.select("taxi_zone_id").distinct().collect()}
assert actual_zones == ZONES, f"Zones inattendues : {actual_zones}"
assert df_gold.filter(F.col(TARGET) < 0).limit(1).count() == 0
n_null = df_gold.select(*[F.sum(F.col(x).isNull().cast("int")).alias(x) for x in sorted(required)]).first().asDict()
assert not any(n_null.values()), f"NULL dans Gold : {n_null}"
float_features = [f.name for f in df_gold.schema.fields if f.name in FEATURES and isinstance(f.dataType,(T.FloatType,T.DoubleType))]
if float_features:
    n_nan = df_gold.select(*[F.sum(F.isnan(x).cast("int")).alias(x) for x in float_features]).first().asDict()
    assert not any(n_nan.values()), f"NaN dans Gold : {n_nan}"
print("Lignes Gold :", df_gold.count(), "| Colonnes features :",len(FEATURES))
print("Période :")
df_gold.agg(F.min("hour_local").alias("min"), F.max("hour_local").alias("max")).show()
display(df_gold.select("taxi_zone_id", "hour_local", TARGET).orderBy("hour_local").limit(10))

## 4. Découpage chronologique et VectorAssembler (Spark MLlib)

In [ ]:
# Train : janvier-août, validation : septembre-octobre, test : novembre-décembre.
df_split = df_gold.withColumn(
    "split",
    F.when(F.col("hour_local") < F.lit(TRAIN_END), "train")
     .when(F.col("hour_local") < F.lit(VALIDATION_END), "validation")
     .otherwise("test")
)
display(df_split.groupBy("split").agg(
    F.count("*").alias("rows"), F.min("hour_local").alias("first_hour"),
    F.max("hour_local").alias("last_hour")
).orderBy("split"))

assembler = VectorAssembler(inputCols=FEATURES, outputCol="features", handleInvalid="error")
base_cols = ["taxi_zone_id", "hour_local", "day_of_week", "hour_of_day", "features", "label"]
def prepare(split_name):
    raw = df_split.filter(F.col("split")==split_name).withColumn("label", F.col(TARGET).cast("double"))
    return assembler.transform(raw).select(*base_cols).cache()

train = prepare("train")
validation = prepare("validation")
test = prepare("test")
counts = [train.count(), validation.count(), test.count()]
assert min(counts)>0, f"Split vide : {counts}"
assert train.agg(F.max("hour_local")).first()[0] < validation.agg(F.min("hour_local")).first()[0]
assert validation.agg(F.max("hour_local")).first()[0] < test.agg(F.min("hour_local")).first()[0]
print("Train / Validation / Test =",counts)

## 5. Métriques — régression, pas classification

**MAE** : erreur moyenne en pickups ; **RMSE** : pénalise les grosses erreurs ; **WAPE** : pourcentage de l'erreur absolue totale. La prédiction est bornée à zéro (le nombre de pickups ne peut pas être négatif).

In [ ]:
def evaluate_regression(predictions):
    checked = predictions.withColumn("prediction", F.greatest(F.lit(0.0), F.col("prediction").cast("double")))
    metric = RegressionEvaluator(labelCol="label", predictionCol="prediction")
    mae = float(metric.setMetricName("mae").evaluate(checked))
    rmse = float(metric.setMetricName("rmse").evaluate(checked))
    totals = checked.agg(
        F.sum(F.abs(F.col("label")-F.col("prediction"))).alias("absolute_error"),
        F.sum(F.abs("label")).alias("total_actual")
    ).first()
    wape = (100.0 * float(totals.absolute_error) / float(totals.total_actual)) if totals.total_actual else None
    return {"mae": mae, "rmse": rmse, "wape_pct": wape}

def fit_baseline(reference):
    return {
        "full": reference.groupBy("taxi_zone_id", "day_of_week", "hour_of_day").agg(F.avg("label").alias("b_full")),
        "hour": reference.groupBy("taxi_zone_id", "hour_of_day").agg(F.avg("label").alias("b_hour")),
        "zone": reference.groupBy("taxi_zone_id").agg(F.avg("label").alias("b_zone")),
        "global": float(reference.agg(F.avg("label")).first()[0])
    }

def predict_baseline(model, scoring):
    return (scoring
        .join(F.broadcast(model["full"]), ["taxi_zone_id", "day_of_week", "hour_of_day"], "left")
        .join(F.broadcast(model["hour"]), ["taxi_zone_id", "hour_of_day"], "left")
        .join(F.broadcast(model["zone"]), ["taxi_zone_id"], "left")
        .withColumn("prediction", F.coalesce("b_full", "b_hour", "b_zone", F.lit(model["global"])))
        .drop("b_full", "b_hour", "b_zone"))

## 6. Entraîner la baseline sur Train

In [ ]:
baseline = fit_baseline(train)
validation_baseline = predict_baseline(baseline, validation)
score_baseline = evaluate_regression(validation_baseline)
print("Baseline | Validation :", score_baseline)

## 7. Entraîner Random Forest (MLlib) sur Train

In [ ]:
rf_params = {"numTrees":100, "maxDepth":10, "minInstancesPerNode":3,
             "featureSubsetStrategy":"auto", "seed":SEED}
rf = RandomForestRegressor(labelCol="label", featuresCol="features", **rf_params)
rf_model = rf.fit(train)
validation_rf = rf_model.transform(validation)
score_rf = evaluate_regression(validation_rf)
print("Random Forest | Validation :", score_rf)

## 8. Entraîner GBTRegressor (MLlib) sur Train

In [ ]:
gbt_params = {"maxIter":60, "maxDepth":5, "stepSize":0.08, "maxBins":64, "seed":SEED}
gbt = GBTRegressor(labelCol="label", featuresCol="features", **gbt_params)
gbt_model = gbt.fit(train)
validation_gbt = gbt_model.transform(validation)
score_gbt = evaluate_regression(validation_gbt)
print("GBT | Validation :", score_gbt)

## 9. Comparer les modèles, sélectionner selon la MAE validation

In [ ]:
validation_results = [
    {"model": "Baseline", **score_baseline},
    {"model": "RandomForest_MLlib", **score_rf},
    {"model": "GBT_MLlib", **score_gbt},
]
display(spark.createDataFrame(validation_results).orderBy("mae"))
selected_model_name = min(validation_results, key=lambda r: r["mae"])["model"]
print("Modèle retenu à partir de VALIDATION uniquement :",selected_model_name)

## 10. Enregistrer les scores de validation dans MLflow

In [ ]:
mlflow.set_experiment(EXPERIMENT_NAME)
VALIDATION_RUN_IDS = {}
for result in validation_results:
    with mlflow.start_run(run_name="validation_" + result["model"]) as run:
        mlflow.set_tags({"project":"NYC_Taxi_Demand", "stage":"validation", "framework":"Spark_MLlib"})
        mlflow.log_params({"model":result["model"], "feature_count":len(FEATURES),
                           "train_before":TRAIN_END, "validation_before":VALIDATION_END, "seed":SEED})
        mlflow.log_metrics({"val_mae":result["mae"], "val_rmse":result["rmse"]})
        if result["wape_pct"] is not None:
            mlflow.log_metric("val_wape_pct", result["wape_pct"])
        VALIDATION_RUN_IDS[result["model"]] = run.info.run_id
print("Runs validation:", VALIDATION_RUN_IDS)

## 11. Réentraîner le meilleur modèle sur Train + Validation

Le test reste intact jusqu'à l'évaluation finale. Les paramètres des modèles ne sont **pas ajustés à partir du test**.

In [ ]:
trainval = train.unionByName(validation).cache()
print("Train+Validation :",trainval.count())

if selected_model_name == "Baseline":
    final_model = None
    final_baseline = fit_baseline(trainval)
    test_predictions = predict_baseline(final_baseline, test)
elif selected_model_name == "RandomForest_MLlib":
    final_model = RandomForestRegressor(labelCol="label", featuresCol="features", **rf_params).fit(trainval)
    test_predictions = final_model.transform(test)
else:
    final_model = GBTRegressor(labelCol="label", featuresCol="features", **gbt_params).fit(trainval)
    test_predictions = final_model.transform(test)

score_test = evaluate_regression(test_predictions)
print("Meilleur modèle (validation):",selected_model_name)
print("TEST NOV–DÉC (évaluation finale):",score_test)

## 12. Sauvegarder le modèle dans OneLake (indépendamment de MLflow)

Le chemin `Files/models/...` appartient au Lakehouse par défaut. S'il s'agit de la baseline, les tables de moyennes sont sauvegardées en Delta. Le contrat JSON sera enregistré aussi comme artefact MLflow lorsque l'upload est disponible.

In [ ]:
spark.sql("CREATE SCHEMA IF NOT EXISTS gold")
model_contract = {
    "model": selected_model_name,
    "features": FEATURES,
    "target": TARGET,
    "train_end_exclusive": TRAIN_END,
    "validation_end_exclusive": VALIDATION_END,
    "prediction_minimum": 0.0,
    "model_location": MODEL_PATH if final_model is not None else "Tables/gold/ml_baseline_v2_*",
    "warnings": ["DST : heures locales naïves", "nettoyage Silver en complete-case",
                 "évaluation sur météo historique, pas forcément prévision météo telle que disponible à l'instant"]
}

if final_model is not None:
    final_model.write().overwrite().save(MODEL_PATH)
    print("Modèle Spark enregistré :", MODEL_PATH)
else:
    for name,key in [("full","full"),("hour","hour"),("zone","zone")]:
        final_baseline[key].write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable(f"gold.ml_baseline_v2_{name}")
    model_contract["baseline_global_mean"] = final_baseline["global"]
    print("Tables baseline enregistrées en Delta")

print("Contrat modèle:")
print(json.dumps(model_contract, indent=2, ensure_ascii=False))

## 13. Run final MLflow — sécuriser l'enregistrement d'artefacts

Les métriques sont enregistrées de manière standard. Le contrat JSON est envoyé **uniquement si le test d'upload a réussi**. En cas de nouvelle erreur de stockage, le modèle natif OneLake reste enregistré.

In [ ]:
with mlflow.start_run(run_name="final_" + selected_model_name) as run:
    mlflow.set_tags({"project":"NYC_Taxi_Demand", "stage":"test", "framework":"Spark_MLlib",
                     "deployment_status":"candidate_not_production"})
    mlflow.log_params({"selected_model":selected_model_name, "feature_count":len(FEATURES),
                       "seed":SEED, "onelake_model_path":model_contract["model_location"]})
    mlflow.log_metrics({"test_mae":score_test["mae"], "test_rmse":score_test["rmse"]})
    if score_test["wape_pct"] is not None:
        mlflow.log_metric("test_wape_pct", score_test["wape_pct"])
    FINAL_RUN_ID = run.info.run_id
    if ARTIFACT_UPLOAD_OK:
        try:
            with tempfile.TemporaryDirectory() as tmp:
                contract_path = Path(tmp)/"model_contract.json"
                contract_path.write_text(json.dumps(model_contract, indent=2, ensure_ascii=False), encoding="utf-8")
                mlflow.log_artifact(str(contract_path), artifact_path="metadata")
            print("Contrat JSON stocké dans MLflow")
        except Exception as exc:
            print("WARN : log_artifact a échoué, modèle natif OneLake déjà enregistré :",type(exc).__name__,str(exc)[:500])
    else:
        print("Upload MLflow désactivé (smoke test non réussi).")
print("FINAL RUN:", FINAL_RUN_ID)

## 14. Vérifier les résultats et analyser les erreurs par zone

In [ ]:
test_checked = (test_predictions
    .withColumn("prediction", F.greatest(F.lit(0.0),F.col("prediction")))
    .withColumn("abs_error", F.abs(F.col("label") - F.col("prediction"))))
display(test_checked.groupBy("taxi_zone_id").agg(
    F.count("*").alias("hours"),
    F.avg("label").alias("actual_mean"),
    F.avg("prediction").alias("predicted_mean"),
    F.avg("abs_error").alias("mae")
).orderBy("taxi_zone_id"))

experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
runs_df = mlflow.search_runs(experiment_ids=[experiment.experiment_id])
columns = [x for x in ["run_id", "status", "tags.mlflow.runName", "metrics.val_mae", "metrics.test_mae", "metrics.test_rmse", "metrics.test_wape_pct"] if x in runs_df.columns]
display(runs_df[columns].head(20))

## 15. Optionnel — écrire les prédictions historiques de test en Delta

In [ ]:
if WRITE_TEST_PREDICTIONS:
    (test_checked.select(
        "taxi_zone_id", "hour_local",
        F.col("label").cast("long").alias("actual_pickups"),
        F.col("prediction").alias("predicted_pickups"),
        "abs_error"
    ).write.format("delta").mode("overwrite").option("overwriteSchema","true")
     .saveAsTable("gold.ml_test_predictions"))
    print("Création réussie : gold.ml_test_predictions")
    spark.sql("SELECT COUNT(*) AS n FROM gold.ml_test_predictions").show()
else:
    print("Écriture Delta des prédictions désactivée")

## 16. Fin et limites à documenter

- MLflow : comparer les runs **dans l'élément Expérience de ton Workspace Fabric**, et non en ouvrant l'URL `api.fabric.microsoft.com/.../mlflow/#/...` dans le navigateur.
- Si l'upload des artefacts renvoie de nouveau `403`, vérifier le compte et la session, puis relever l'ID du run et contacter le support Fabric si nécessaire. Ne pas rajouter de rôles administrateurs : le compte possède déjà le rôle Workspace Admin.
- `gold.ml_test_predictions` contient des **prédictions historiques de novembre–décembre 2025**, pas des prévisions opérationnelles en temps réel.
- Les prédictions futures doivent utiliser les mêmes features et de **véritables prévisions météo disponibles au moment de la prédiction** ; valider l'effet du DST et des trajets exclus dans Silver.
- Pour la production, ajouter orchestration, monitoring, qualité, modèle chargé pour inférence et Power BI.